#Install dependencies


In [ ]:

!pip install requests pandas pyarrow fastparquet
#  Imports
import requests
import pandas as pd
import os
import time

In [ ]:

# Configuration
DATASET_ID = "ijzp-q8t2"
BASE_URL = f"https://data.cityofchicago.org/resource/{DATASET_ID}.csv"

# We want 2010–2022 only
START_YEAR = 2008
END_YEAR = 2022
# Socrata API returns max 50,000 rows per request — we page through
CHUNK_SIZE = 50000
OUTPUT_DIR = "/content/chicago_crimes"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Download year by year
all_dfs = []

for year in range(START_YEAR, END_YEAR + 1):
    print(f"\nDownloading year: {year}")
    year_chunks = []
    offset = 0

    while True:
        # SoQL query: filter by year, paginate with limit + offset
        params = {
            "$where": f"year={year}",
            "$limit": CHUNK_SIZE,
            "$offset": offset,
            "$order": ":id"
        }

        response = requests.get(BASE_URL, params=params, timeout=120)

        if response.status_code != 200:
            print(f"   Error {response.status_code} at offset {offset}")
            break

        # Read CSV chunk
        from io import StringIO
        chunk_df = pd.read_csv(StringIO(response.text))

        if len(chunk_df) == 0:
            print(f"   Year {year} complete — {offset} rows total")
            break

        year_chunks.append(chunk_df)
        offset += len(chunk_df)
        print(f"  ... fetched {offset} rows so far", end="\r")

        # Small delay to avoid rate limiting
        time.sleep(0.5)

    if year_chunks:
        year_df = pd.concat(year_chunks, ignore_index=True)

        # Save each year as its own Parquet file
        year_path = f"{OUTPUT_DIR}/crimes_{year}.parquet"
        year_df.to_parquet(year_path, index=False)
        print(f"   Saved: crimes_{year}.parquet  ({len(year_df):,} rows)")
        all_dfs.append(year_df)


   Year 2008 complete — 427216 rows total
   Saved: crimes_2008.parquet  (427,216 rows)

   Year 2009 complete — 392867 rows total
   Saved: crimes_2009.parquet  (392,867 rows)

   Year 2010 complete — 370566 rows total
   Saved: crimes_2010.parquet  (370,566 rows)

   Year 2011 complete — 352052 rows total
   Saved: crimes_2011.parquet  (352,052 rows)

   Year 2012 complete — 336374 rows total
   Saved: crimes_2012.parquet  (336,374 rows)

   Year 2013 complete — 307612 rows total
   Saved: crimes_2013.parquet  (307,612 rows)

   Year 2014 complete — 275880 rows total
   Saved: crimes_2014.parquet  (275,880 rows)

   Year 2015 complete — 264887 rows total
   Saved: crimes_2015.parquet  (264,887 rows)

   Year 2016 complete — 269959 rows total
   Saved: crimes_2016.parquet  (269,959 rows)

   Year 2017 complete — 269285 rows total
   Saved: crimes_2017.parquet  (269,285 rows)

   Year 2018 complete — 269147 rows total
   Saved: crimes_2018.parquet  (269,147 rows)

   Year 2019 complet

In [ ]:
# Combine and save as CSV FIRST
print("\n Combining all years")
full_df = pd.concat(all_dfs, ignore_index=True)

# Save as CSV first
csv_path = f"{OUTPUT_DIR}/chicago_crimes_full.csv"
print(" Saving as CSV ")
full_df.to_csv(csv_path, index=False)


 Combining all years
 Saving as CSV 


In [ ]:
#  Show CSV size FIRST
def get_size(path):
    size = os.path.getsize(path)
    if size >= 1e9:
        return f"{size/1e9:.2f} GB"
    return f"{size/1e6:.0f} MB"

csv_size = os.path.getsize(csv_path)
print(" RAW DATASET (CSV)")
print("_"*20)
print(f"  Total Rows     : {len(full_df):,}")
print(f"  Total Columns  : {len(full_df.columns)}")
print(f"  CSV Size       : {get_size(csv_path)}")
print(f"  Requirement    : ≥ 1 GB  -->  {' PASSED' if csv_size >= 1e9 else ' NOT MET'}")
print(f"  Years Covered  : {START_YEAR}–{END_YEAR}")

 RAW DATASET (CSV)
____________________
  Total Rows     : 4,459,898
  Total Columns  : 22
  CSV Size       : 1.12 GB
  Requirement    : ≥ 1 GB  -->   PASSED
  Years Covered  : 2008–2022


In [ ]:
# NOW convert CSV --> Parquet with justification
print("\n Converting CSV → Parquet (Snappy compression)")
print("   Reason: Parquet offers columnar storage, predicate")

parquet_path = f"{OUTPUT_DIR}/chicago_crimes_full.parquet"
full_df.to_parquet(parquet_path, index=False, compression='snappy')

parquet_size = os.path.getsize(parquet_path)
compression_ratio = csv_size / parquet_size

print(" AFTER CONVERSION T")
print(f"  CSV Size       : {get_size(csv_path)}")
print(f"  Parquet Size   : {get_size(parquet_path)}")
print(f"  Compression    : {compression_ratio:.1f}x smaller")
print(f"  Space Saved    : {get_size.__doc__}")
space_saved = (1 - parquet_size/csv_size) * 100
print(f"  Space Saved    : {space_saved:.1f}%")


 Converting CSV → Parquet (Snappy compression)
   Reason: Parquet offers columnar storage, predicate
 AFTER CONVERSION T
  CSV Size       : 1.12 GB
  Parquet Size   : 280 MB
  Compression    : 4.0x smaller
  Space Saved    : None
  Space Saved    : 75.1%


In [ ]:
#  Final preview
print(f"\n Columns ({len(full_df.columns)}):")
for i, col in enumerate(full_df.columns, 1):
    print(f"  {i:2}. {col}")

print(f"\n Target column — Arrest distribution:")
print(full_df['arrest'].value_counts())

print(f"\n Records per year:")
print(full_df['year'].value_counts().sort_index().to_string())


 Columns (22):
   1. id
   2. case_number
   3. date
   4. block
   5. iucr
   6. primary_type
   7. description
   8. location_description
   9. arrest
  10. domestic
  11. beat
  12. district
  13. ward
  14. community_area
  15. fbi_code
  16. x_coordinate
  17. y_coordinate
  18. year
  19. updated_on
  20. latitude
  21. longitude
  22. location

 Target column — Arrest distribution:
arrest
False    3410031
True     1049867
Name: count, dtype: int64

 Records per year:
year
2008    427216
2009    392867
2010    370566
2011    352052
2012    336374
2013    307612
2014    275880
2015    264887
2016    269959
2017    269285
2018    269147
2019    261701
2020    212700
2021    209654
2022    239998


In [ ]:
#  Quick preview
print("\n Column names:")
print(full_df.columns.tolist())
print("\n First 3 rows:")
full_df.head(3)


 Column names:
['id', 'case_number', 'date', 'block', 'iucr', 'primary_type', 'description', 'location_description', 'arrest', 'domestic', 'beat', 'district', 'ward', 'community_area', 'fbi_code', 'x_coordinate', 'y_coordinate', 'year', 'updated_on', 'latitude', 'longitude', 'location']

 First 3 rows:


,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,...,ward,community_area,fbi_code,x_coordinate,y_coordinate,year,updated_on,latitude,longitude,location
0,10001244,HY190838,2008-05-01T12:00:00.000,013XX W 76TH ST,1153,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,APARTMENT,False,False,...,17.0,71.0,11,NaN,NaN,2008,2015-08-17T15:03:40.000,NaN,NaN,NaN
1,10002056,HY191556,2008-01-22T00:00:00.000,043XX W NORTH AVE,1120,DECEPTIVE PRACTICE,FORGERY,OTHER,False,False,...,37.0,23.0,10,NaN,NaN,2008,2015-08-17T15:03:40.000,NaN,NaN,NaN
2,10010211,HY199673,2008-01-06T00:00:00.000,115XX S STEWART AVE,1153,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,RESIDENCE,False,False,...,34.0,53.0,11,NaN,NaN,2008,2015-08-17T15:03:40.000,NaN,NaN,NaN


In [ ]:
# DATA INSPECTION
import pandas as pd
import numpy as np

print("SHAPE & BASIC INFO")

print(f"Rows    : {full_df.shape[0]:,}")
print(f"Columns : {full_df.shape[1]}")


SHAPE & BASIC INFO
Rows    : 4,459,898
Columns : 22


In [ ]:
print("DATA TYPES")

print(full_df.dtypes)


print("MISSING VALUES")

missing = full_df.isnull().sum()
missing_pct = (missing / len(full_df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0])


DATA TYPES
id                        int64
case_number              object
date                     object
block                    object
iucr                     object
primary_type             object
description              object
location_description     object
arrest                     bool
domestic                   bool
beat                      int64
district                float64
ward                    float64
community_area          float64
fbi_code                 object
x_coordinate            float64
y_coordinate            float64
year                      int64
updated_on               object
latitude                float64
longitude               float64
location                 object
dtype: object
MISSING VALUES
                      Missing Count  Missing %
location_description          11711       0.26
district                         41       0.00
ward                            103       0.00
community_area                 1089       0.02
x_coordinate         

In [ ]:
print("BASIC STATISTICS (Numerical Columns)")

print(full_df.describe())


print("CATEGORICAL COLUMNS — Unique Value Counts")

cat_cols = full_df.select_dtypes(include='object').columns
for col in cat_cols:
    print(f"  {col:25s}: {full_df[col].nunique():>6,} unique values")


BASIC STATISTICS (Numerical Columns)
                 id          beat      district          ward  community_area  \
count  4.459898e+06  4.459898e+06  4.459857e+06  4.459795e+06    4.458809e+06   
mean   9.513411e+06  1.168214e+03  1.128478e+01  2.293434e+01    3.738747e+01   
std    2.059232e+06  6.996400e+02  6.946749e+00  1.384505e+01    2.153463e+01   
min    4.379000e+03  1.110000e+02  1.000000e+00  1.000000e+00    0.000000e+00   
25%    7.756164e+06  6.140000e+02  6.000000e+00  1.000000e+01    2.300000e+01   
50%    9.532172e+06  1.032000e+03  1.000000e+01  2.300000e+01    3.200000e+01   
75%    1.130898e+07  1.723000e+03  1.700000e+01  3.400000e+01    5.600000e+01   
max    1.412152e+07  2.535000e+03  3.100000e+01  5.000000e+01    7.700000e+01   

       x_coordinate  y_coordinate          year      latitude     longitude  
count  4.401269e+06  4.401269e+06  4.459898e+06  4.401269e+06  4.401269e+06  
mean   1.164605e+06  1.885679e+06  2.014138e+03  4.184190e+01 -8.767148e+01  

In [ ]:
print("TARGET COLUMN — Arrest Balance")

arrest_counts = full_df['arrest'].value_counts()
arrest_pct = full_df['arrest'].value_counts(normalize=True) * 100
for val in arrest_counts.index:
    print(f"  {str(val):6s}: {arrest_counts[val]:>10,}  ({arrest_pct[val]:.1f}%)")

print("DUPLICATE ROWS CHECK")

dupes = full_df.duplicated().sum()
print(f"  Duplicate rows: {dupes:,}")


TARGET COLUMN — Arrest Balance
  False :  3,410,031  (76.5%)
  True  :  1,049,867  (23.5%)
DUPLICATE ROWS CHECK
  Duplicate rows: 0


In [ ]:
print("SAMPLE VALUE CHECK (Key Columns)")

key_cols = ['primary_type', 'location_description', 'fbi_code', 'district']
for col in key_cols:
    print(f"\n  Top 5 values in '{col}':")
    print(full_df[col].value_counts().head(5).to_string())

SAMPLE VALUE CHECK (Key Columns)

  Top 5 values in 'primary_type':
primary_type
THEFT              978129
BATTERY            803641
CRIMINAL DAMAGE    493536
NARCOTICS          366084
ASSAULT            300579

  Top 5 values in 'location_description':
location_description
STREET       1062559
RESIDENCE     731241
APARTMENT     592077
SIDEWALK      436280
OTHER         142484

  Top 5 values in 'fbi_code':
fbi_code
06     978130
08B    691045
14     493536
18     366072
26     312368

  Top 5 values in 'district':
district
8.0     297522
11.0    295068
6.0     270767
4.0     258341
7.0     253707


In [ ]:
# PYSPARK SETUP AND DATA INGESTION
# Install required packages
!pip install pyspark findspark --quiet

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# SparkSession Configuration with justified settings
spark = SparkSession.builder \
    .appName("ChicagoCrimesML") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .config("spark.sql.shuffle.partitions", "100") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("SparkSession created successfully")
print("Spark Version :", spark.version)
print("App Name      :", spark.sparkContext.appName)
print("Driver Memory : 8g")
print("Shuffle Parts : 100")
print("\nSpark UI available at:", spark.sparkContext.uiWebUrl)

SparkSession created successfully
Spark Version : 4.0.2
App Name      : ChicagoCrimesML
Driver Memory : 8g
Shuffle Parts : 100

Spark UI available at: http://09b9a4b27e60:4040


In [ ]:
# Load Parquet into Spark DataFrame

PARQUET_PATH = "/content/chicago_crimes/chicago_crimes_full.parquet"

df = spark.read.parquet(PARQUET_PATH)

print("\nData loaded successfully")
print("Rows    :", df.count())
print("Columns :", len(df.columns))


Data loaded successfully
Rows    : 4459898
Columns : 22


In [ ]:

# Define explicit schema for validation

print("\nSchema:")
df.printSchema()


#  Data Validation at Ingestion

print("\nData Validation Report")

total_rows = df.count()
print(f"Total rows         : {total_rows:,}")
print(f"Total columns      : {len(df.columns)}")
print(f"Null in arrest     : {df.filter(F.col('arrest').isNull()).count()}")
print(f"Null in year       : {df.filter(F.col('year').isNull()).count()}")
print(f"Year range         : {df.agg(F.min('year')).collect()[0][0]} - {df.agg(F.max('year')).collect()[0][0]}")
print(f"Duplicate IDs      : {total_rows - df.select('id').distinct().count()}")


Schema:
root
 |-- id: long (nullable = true)
 |-- case_number: string (nullable = true)
 |-- date: string (nullable = true)
 |-- block: string (nullable = true)
 |-- iucr: string (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- location_description: string (nullable = true)
 |-- arrest: boolean (nullable = true)
 |-- domestic: boolean (nullable = true)
 |-- beat: long (nullable = true)
 |-- district: double (nullable = true)
 |-- ward: double (nullable = true)
 |-- community_area: double (nullable = true)
 |-- fbi_code: string (nullable = true)
 |-- x_coordinate: double (nullable = true)
 |-- y_coordinate: double (nullable = true)
 |-- year: long (nullable = true)
 |-- updated_on: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- location: string (nullable = true)


Data Validation Report
Total rows         : 4,459,898
Total columns      : 22
Null in arrest     : 0

In [ ]:
# Partitioning Strategy

PARTITIONED_PATH = "/content/chicago_crimes/partitioned"

df.write \
  .mode("overwrite") \
  .partitionBy("year") \
  .parquet(PARTITIONED_PATH)

print("\nPartitioned data saved by year")
print("Path:", PARTITIONED_PATH)
print("Justification: Partitioning by year aligns with temporal")
print("query patterns and enables partition pruning in Spark.")

# Reload partitioned data
df = spark.read.parquet(PARTITIONED_PATH)
df.cache()
print("\nDataFrame cached for repeated use")
print("Rows after reload:", df.count())


Partitioned data saved by year
Path: /content/chicago_crimes/partitioned
Justification: Partitioning by year aligns with temporal
query patterns and enables partition pruning in Spark.

DataFrame cached for repeated use
Rows after reload: 4459898
